# Занятие 3_3_3. Инструменты для сбора данных. Пример 3

## Scrapy, SQLAlchemy, pandas и Peewee

В этом примере мы соберём товары с двух локальных HTML-страниц, сохраним результат в SQLite и прочитаем его двумя способами.

Рабочая цепочка:

`HTML-страницы → Scrapy → JSONL → pandas → SQLAlchemy → SQLite → Peewee`

Интернет не требуется: Scrapy обращается к локальному учебному сайту.

## Что получится в результате

Будут созданы:

- `результаты_пример_3/товары_scrapy.jsonl`;
- `результаты_пример_3/каталог.sqlite`;
- `результаты_пример_3/дорогие_товары_peewee.csv`.

## Подготовка рабочей среды

Запустите следующие две ячейки **до всех остальных**.

Первая ячейка проверяет наличие библиотек и устанавливает только отсутствующие пакеты. Вторая ячейка выполняет все импорты. После этого notebook нужно запускать сверху вниз командой **«Выполнить все»**.

Scrapy, SQLAlchemy и Peewee могут отсутствовать в новом окружении Python. Если компьютер работает без доступа к интернету, установите зависимости заранее из файла `requirements.txt`:

```bash
python -m pip install -r requirements.txt
```


In [ ]:
# Подключаем стандартные модули Python для проверки и установки библиотек.
import importlib.util
import subprocess
import sys

# Сопоставляем имя модуля в Python и имя пакета для установки через pip.
необходимые_пакеты = {
    "pandas": "pandas>=2.0,<4",
    "scrapy": "Scrapy>=2.13,<3",
    "sqlalchemy": "SQLAlchemy>=2.0,<3",
    "peewee": "peewee>=3.17,<5",
}

# Создаём пустой список для отсутствующих пакетов.
отсутствующие_пакеты = []

# Проверяем каждый модуль по очереди.
for имя_модуля, пакет_для_установки in необходимые_пакеты.items():
    # Если модуль не найден, добавляем соответствующий пакет в список.
    if importlib.util.find_spec(имя_модуля) is None:
        отсутствующие_пакеты.append(пакет_для_установки)

# Устанавливаем только отсутствующие пакеты.
if отсутствующие_пакеты:
    # Показываем список пакетов перед установкой.
    print("Будут установлены пакеты:", отсутствующие_пакеты)

    try:
        # Запускаем pip через тот же интерпретатор, что и текущий notebook.
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                *отсутствующие_пакеты,
            ]
        )
        # Сообщаем об успешной установке.
        print("Установка завершена. Можно выполнять следующую ячейку.")

    except subprocess.CalledProcessError as ошибка:
        # Выводим понятное сообщение при невозможности установки.
        raise RuntimeError(
            "Не удалось установить библиотеки. "
            "Проверьте доступ к интернету или заранее выполните команду "
            "python -m pip install -r requirements.txt"
        ) from ошибка
else:
    # Сообщаем, что установка не требуется.
    print("Все необходимые библиотеки уже установлены.")


In [ ]:
# Подключаем Path для работы с файлами и папками.
from pathlib import Path

# Подключаем subprocess для запуска Scrapy как отдельной команды.
import subprocess

# Подключаем sys, чтобы использовать тот же интерпретатор Python.
import sys

# Подключаем threading для локального HTTP-сервера.
import threading

# Подключаем встроенный HTTP-сервер Python.
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler

# Подключаем partial для передачи серверу папки сайта.
from functools import partial

# Подключаем pandas для чтения и записи таблиц.
import pandas as pd

# Подключаем модуль SQLAlchemy и функции для работы с базой данных.
import sqlalchemy
from sqlalchemy import create_engine, inspect

# Подключаем модуль Peewee и нужные классы ORM.
import peewee
from peewee import Model, SqliteDatabase, TextField, IntegerField

# Подключаем Scrapy для сбора данных с HTML-страниц.
import scrapy

# Явно подключаем display для отображения таблиц в notebook.
from IPython.display import display

# Показываем версии основных библиотек.
print("pandas:", pd.__version__)
print("Scrapy:", scrapy.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("Peewee:", peewee.__version__)

# Сообщаем, что импорт завершён успешно.
print("Все библиотеки успешно импортированы.")


## 1. Создаём рабочие папки

In [ ]:
# Получаем текущую рабочую папку.
рабочая_папка = Path.cwd()

# Создаём путь к локальному учебному сайту.
папка_сайта = рабочая_папка / "учебный_сайт_пример_3"

# Создаём путь к папке результатов.
папка_результатов = рабочая_папка / "результаты_пример_3"

# Создаём папку сайта.
папка_сайта.mkdir(exist_ok=True)

# Создаём папку результатов.
папка_результатов.mkdir(exist_ok=True)

# Показываем пути.
print("Папка сайта:", папка_сайта)
print("Папка результатов:", папка_результатов)

## 2. Создаём первую HTML-страницу

На первой странице находятся два товара и ссылка на следующую страницу.

In [ ]:
# Создаём HTML-код первой страницы.
html_страница_1 = """
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Каталог — страница 1</title>
</head>
<body>
    <div class="product">
        <span class="code">Т-101</span>
        <span class="name">Мышь</span>
        <span class="price">2500</span>
        <span class="stock">12</span>
    </div>

    <div class="product">
        <span class="code">Т-102</span>
        <span class="name">Клавиатура</span>
        <span class="price">4200</span>
        <span class="stock">4</span>
    </div>

    <a class="next" href="страница_2.html">Следующая страница</a>
</body>
</html>
"""

# Создаём путь к первой странице.
путь_страницы_1 = папка_сайта / "страница_1.html"

# Сохраняем первую страницу.
путь_страницы_1.write_text(html_страница_1, encoding="utf-8")

# Показываем путь.
print("Создан файл:", путь_страницы_1)

## 3. Создаём вторую HTML-страницу

In [ ]:
# Создаём HTML-код второй страницы.
html_страница_2 = """
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Каталог — страница 2</title>
</head>
<body>
    <div class="product">
        <span class="code">Т-103</span>
        <span class="name">Наушники</span>
        <span class="price">5600</span>
        <span class="stock">0</span>
    </div>

    <div class="product">
        <span class="code">Т-104</span>
        <span class="name">Веб-камера</span>
        <span class="price">3800</span>
        <span class="stock">7</span>
    </div>
</body>
</html>
"""

# Создаём путь ко второй странице.
путь_страницы_2 = папка_сайта / "страница_2.html"

# Сохраняем вторую страницу.
путь_страницы_2.write_text(html_страница_2, encoding="utf-8")

# Показываем путь.
print("Создан файл:", путь_страницы_2)

## 4. Запускаем локальный HTTP-сервер

Scrapy будет обращаться к этому серверу как к небольшому сайту.

In [ ]:
# Создаём обработчик запросов для папки сайта.
обработчик = partial(SimpleHTTPRequestHandler, directory=str(папка_сайта))

# Создаём сервер и просим Python выбрать свободный порт.
локальный_сервер = ThreadingHTTPServer(("127.0.0.1", 0), обработчик)

# Получаем номер выбранного порта.
номер_порта = локальный_сервер.server_address[1]

# Создаём отдельный поток для сервера.
поток_сервера = threading.Thread(
    target=локальный_сервер.serve_forever,
    daemon=True
)

# Запускаем сервер.
поток_сервера.start()

# Формируем адрес первой страницы.
стартовый_адрес = f"http://127.0.0.1:{номер_порта}/страница_1.html"

# Показываем адрес.
print("Стартовая страница:", стартовый_адрес)

## 5. Создаём Scrapy-паука

Паук:

1. открывает первую страницу;
2. извлекает товары по CSS-селекторам;
3. находит ссылку `Следующая страница`;
4. переходит по ссылке;
5. извлекает товары со второй страницы.

Scrapy запускается отдельным процессом. Это делает повторный запуск notebook более устойчивым.

In [ ]:
# Формируем текст Python-скрипта Scrapy.
код_паука = f'''import scrapy


class КаталогSpider(scrapy.Spider):
    # Указываем уникальное имя паука.
    name = "учебный_каталог"

    # Указываем первую страницу обхода.
    start_urls = ["{стартовый_адрес}"]

    # Уменьшаем объём служебных сообщений Scrapy.
    custom_settings = {{
        "LOG_LEVEL": "WARNING",
        "FEED_EXPORT_ENCODING": "utf-8"
    }}

    def parse(self, response):
        # Находим все карточки товаров на текущей странице.
        for карточка in response.css("div.product"):
            # Возвращаем один словарь для каждого товара.
            yield {{
                "Код товара": карточка.css("span.code::text").get(),
                "Название товара": карточка.css("span.name::text").get(),
                "Цена": int(карточка.css("span.price::text").get()),
                "Остаток": int(карточка.css("span.stock::text").get()),
                "Страница-источник": response.url
            }}

        # Ищем ссылку на следующую страницу.
        следующая_ссылка = response.css("a.next::attr(href)").get()

        # Если ссылка найдена, просим Scrapy перейти по ней.
        if следующая_ссылка:
            yield response.follow(следующая_ссылка, callback=self.parse)
'''

# Создаём путь к файлу паука.
путь_паука = рабочая_папка / "учебный_паук.py"

# Сохраняем код паука в Python-файл.
путь_паука.write_text(код_паука, encoding="utf-8")

# Показываем путь к скрипту.
print("Создан Scrapy-скрипт:", путь_паука)

## 6. Запускаем Scrapy и сохраняем результат

Формат JSONL означает: один JSON-объект в каждой строке файла.

In [ ]:
# Создаём путь к выходному JSONL-файлу.
путь_scrapy_jsonl = папка_результатов / "товары_scrapy.jsonl"

# Удаляем старый результат, если он остался от предыдущего запуска.
if путь_scrapy_jsonl.exists():
    путь_scrapy_jsonl.unlink()

# Формируем команду запуска Scrapy.
команда_scrapy = [
    sys.executable,
    "-m",
    "scrapy",
    "runspider",
    str(путь_паука),
    "-O",
    str(путь_scrapy_jsonl)
]

# Запускаем Scrapy и сохраняем текст служебного вывода.
результат_scrapy = subprocess.run(
    команда_scrapy,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

# Показываем код завершения процесса.
print("Код завершения Scrapy:", результат_scrapy.returncode)

# Если Scrapy завершился с ошибкой, показываем последние сообщения.
if результат_scrapy.returncode != 0:
    print(результат_scrapy.stderr[-2000:])

# Проверяем, что процесс завершился успешно.
assert результат_scrapy.returncode == 0, "Scrapy завершился с ошибкой"

# Проверяем, что файл был создан.
assert путь_scrapy_jsonl.exists(), "Scrapy не создал выходной файл"

# Показываем путь к результату.
print("Scrapy сохранил данные:", путь_scrapy_jsonl)

## 7. Загружаем результат Scrapy в pandas

In [ ]:
# Читаем JSONL-файл в DataFrame.
таблица_scrapy = pd.read_json(
    путь_scrapy_jsonl,
    lines=True
)

# Показываем полученную таблицу.
display(таблица_scrapy)

# Проверяем ожидаемое количество строк.
assert len(таблица_scrapy) == 4, "Ожидалось четыре товара"

## 8. Сохраняем DataFrame в SQLite через SQLAlchemy

`create_engine()` создаёт объект подключения к базе. Метод pandas `to_sql()` записывает DataFrame в таблицу базы данных.

In [ ]:
# Создаём путь к файлу базы данных SQLite.
путь_базы = папка_результатов / "каталог.sqlite"

# Удаляем старую базу для воспроизводимого повторного запуска.
if путь_базы.exists():
    путь_базы.unlink()

# Формируем строку подключения SQLAlchemy.
строка_подключения = f"sqlite+pysqlite:///{путь_базы.as_posix()}"

# Создаём SQLAlchemy Engine.
движок_sqlalchemy = create_engine(строка_подключения)

# Записываем DataFrame в таблицу SQLite.
таблица_scrapy.to_sql(
    "Товары",
    движок_sqlalchemy,
    if_exists="replace",
    index=False
)

# Получаем список таблиц базы.
список_таблиц = inspect(движок_sqlalchemy).get_table_names()

# Показываем созданные таблицы.
print("Таблицы базы:", список_таблиц)

# Проверяем наличие таблицы Товары.
assert "Товары" in список_таблиц, "Таблица Товары не создана"

## 9. Читаем таблицу через pandas и SQLAlchemy

In [ ]:
# Загружаем SQL-таблицу обратно в DataFrame.
таблица_из_sql = pd.read_sql_table(
    "Товары",
    движок_sqlalchemy
)

# Показываем данные, прочитанные из SQLite.
display(таблица_из_sql)

# Проверяем количество строк.
assert len(таблица_из_sql) == 4, "В базе должно быть четыре товара"

## 10. Подключаем Peewee к той же базе

Peewee — это компактная ORM. Модель Python связывает атрибуты класса со столбцами таблицы.

Здесь Peewee не создаёт новую таблицу: он читает таблицу `Товары`, которую ранее создали pandas и SQLAlchemy.

> Названия столбцов в SQLite остаются русскими. Для атрибутов ORM-модели используем короткие латинские технические имена. Это делает пример устойчивым в разных версиях Peewee и не меняет названия столбцов в итоговых таблицах.


In [ ]:
# Создаём подключение Peewee к существующему SQLite-файлу.
база_peewee = SqliteDatabase(путь_базы)

# Создаём базовую модель.
class БазоваяМодель(Model):
    # Связываем все модели с одной базой данных.
    class Meta:
        database = база_peewee


# Описываем модель существующей таблицы Товары.
class ProductModel(БазоваяМодель):
    # Имя атрибута модели пишем латиницей.
    # Параметр column_name связывает его с русским столбцом Код товара.
    product_code = TextField(
        column_name="Код товара",
        primary_key=True
    )

    # Связываем технический атрибут с русским столбцом Название товара.
    product_name = TextField(
        column_name="Название товара"
    )

    # Связываем технический атрибут с русским столбцом Цена.
    price = IntegerField(
        column_name="Цена"
    )

    # Связываем технический атрибут с русским столбцом Остаток.
    stock = IntegerField(
        column_name="Остаток"
    )

    # Связываем технический атрибут с русским столбцом Страница-источник.
    source_page = TextField(
        column_name="Страница-источник"
    )

    # Указываем настоящее имя таблицы в SQLite.
    class Meta:
        table_name = "Товары"
        database = база_peewee


# Открываем соединение с базой.
база_peewee.connect(reuse_if_open=True)

# Показываем таблицы, доступные через Peewee.
print("Таблицы Peewee:", база_peewee.get_tables())

# Проверяем, что нужная таблица доступна.
assert "Товары" in база_peewee.get_tables(), "Peewee не видит таблицу Товары"


## 11. Выполняем запрос через Peewee

Получим товары с ценой больше 3000. Это пример чтения собранных данных из базы, а не отдельный блок анализа.

In [ ]:
# Создаём Peewee-запрос.
запрос_peewee = (
    ProductModel
    .select()
    .where(ProductModel.price > 3000)
    .order_by(ProductModel.price)
)

# Создаём пустой список для результатов запроса.
строки_peewee = []

# Перебираем объекты, возвращённые ORM.
for товар in запрос_peewee:
    # Преобразуем каждый объект модели в обычный словарь.
    # В итоговом словаре снова используем понятные русские названия.
    строки_peewee.append({
        "Код товара": товар.product_code,
        "Название товара": товар.product_name,
        "Цена": товар.price,
        "Остаток": товар.stock,
        "Страница-источник": товар.source_page
    })

# Создаём DataFrame из результата Peewee.
дорогие_товары = pd.DataFrame(строки_peewee)

# Показываем результат.
display(дорогие_товары)

# Проверяем количество найденных товаров.
assert len(дорогие_товары) == 3, "Ожидалось три товара с ценой больше 3000"

# Проверяем, что текстовые поля прочитались из SQLite, а не превратились в None.
текстовые_столбцы = [
    "Код товара",
    "Название товара",
    "Страница-источник"
]

assert not дорогие_товары[текстовые_столбцы].isna().any().any(), (
    "Peewee не прочитал одно или несколько текстовых полей"
)

# Проверяем ожидаемый порядок товаров по возрастанию цены.
assert дорогие_товары["Код товара"].tolist() == [
    "Т-104",
    "Т-102",
    "Т-103"
], "Порядок или состав результата отличается от ожидаемого"


## 12. Сохраняем результат запроса Peewee

In [ ]:
# Создаём путь к CSV-файлу.
путь_дорогих_товаров = папка_результатов / "дорогие_товары_peewee.csv"

# Сохраняем результат запроса.
дорогие_товары.to_csv(
    путь_дорогих_товаров,
    index=False,
    encoding="utf-8-sig"
)

# Показываем путь к файлу.
print("Файл сохранён:", путь_дорогих_товаров)

## 13. Завершаем работу

In [ ]:
# Закрываем соединение Peewee.
if not база_peewee.is_closed():
    база_peewee.close()

# Освобождаем соединения SQLAlchemy.
движок_sqlalchemy.dispose()

# Останавливаем локальный HTTP-сервер.
локальный_сервер.shutdown()

# Закрываем серверный сокет.
локальный_сервер.server_close()

# Сообщаем о завершении.
print("Соединения закрыты. Локальный сервер остановлен.")

## Итоги примера

Мы использовали:

- Scrapy — обход двух связанных HTML-страниц;
- pandas — загрузка JSONL и работа с DataFrame;
- SQLAlchemy — подключение и запись DataFrame в SQLite;
- Peewee — ORM-модель и чтение строк из той же таблицы;
- SQLite — локальное хранилище собранных данных.

### Контрольные вопросы

1. Для чего Scrapy нужен метод `parse()`?
2. Что делает `response.follow()`?
3. Зачем Scrapy запускается отдельным процессом?
4. Для чего SQLAlchemy создаёт Engine?
5. Что связывает модель Peewee с таблицей SQLite?